# 🏥 Healthcare Veri Seti — MVP v2

**Veri Seti:** [Kaggle — Healthcare Dataset (prasad22)](https://www.kaggle.com/datasets/prasad22/healthcare-dataset)  
**Hedef:** Hasta test sonucunu tahmin etmek (Normal / Abnormal / Inconclusive)

## Proje Akışı
1. Kütüphane ve Veri Yükleme  
2. Keşifsel Veri Analizi (EDA)  
3. Veri Ön İşleme  
4. Model Eğitimi ve Test Skoru  
5. Model Kaydetme  
6. Streamlit Uygulaması (UI + Deploy)

## 1. Kütüphane ve Veri Yükleme

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print('Kütüphaneler yüklendi.')

Kütüphaneler yüklendi.


In [4]:
df = pd.read_csv('healthcare_dataset.csv')

print(f'Satır sayısı : {df.shape[0]:,}')
print(f'Sütun sayısı : {df.shape[1]}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'healthcare_dataset.csv'

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
print('=== Değişken Tipleri & Eksik Değerler ===')
info = pd.DataFrame({
    'dtype'       : df.dtypes,
    'null_sayisi' : df.isnull().sum(),
    'null_yuzde'  : (df.isnull().sum() / len(df) * 100).round(2)
})
print(info)

In [ ]:
print('=== Sayısal Değişkenler ===')
print(df.describe())

In [ ]:
print('=== Kategorik Değişkenler ===')
cat_cols = ['Gender', 'Blood Type', 'Medical Condition',
            'Insurance Provider', 'Admission Type', 'Medication', 'Test Results']

for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 3. Veri Ön İşleme

In [ ]:
df_model = df.copy()

# Tarih feature'ları
df_model['Date of Admission'] = pd.to_datetime(df_model['Date of Admission'])
df_model['Discharge Date']    = pd.to_datetime(df_model['Discharge Date'])
df_model['yatis_gun']         = (df_model['Discharge Date'] - df_model['Date of Admission']).dt.days
df_model['admission_ay']      = df_model['Date of Admission'].dt.month

# Label Encoding
le = LabelEncoder()
encode_cols = ['Gender', 'Blood Type', 'Medical Condition',
               'Insurance Provider', 'Admission Type', 'Medication']
for col in encode_cols:
    df_model[col + '_enc'] = le.fit_transform(df_model[col])

# Hedef değişken
df_model['test_target'] = le.fit_transform(df_model['Test Results'])
target_map = dict(zip(le.transform(le.classes_), le.classes_))
print('Sınıf eşleştirmesi:', target_map)

# Feature listesi
features = ['Age', 'yatis_gun', 'Billing Amount', 'admission_ay',
            'Gender_enc', 'Blood Type_enc', 'Medical Condition_enc',
            'Insurance Provider_enc', 'Admission Type_enc', 'Medication_enc']

X = df_model[features]
y = df_model['test_target']

print(f'\nFeature sayısı : {X.shape[1]}')
print(f'Örnek sayısı   : {X.shape[0]:,}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Eğitim seti: {X_train.shape[0]:,} satır')
print(f'Test seti  : {X_test.shape[0]:,} satır')

## 4. Model Eğitimi ve Test Skoru

**Random Forest Classifier** — birden fazla karar ağacının oylamasıyla tahmin üretir. Yüksek doğruluk, düşük overfitting.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)

print('=== Test Sonuçları ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=list(target_map.values())))

## 5. Model Kaydetme

In [ ]:
joblib.dump(model,  'model_classification.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('model_classification.pkl → kaydedildi')
print('scaler.pkl              → kaydedildi')

## 6. Streamlit Uygulaması

Aşağıdaki hücre, `app.py` dosyasını oluşturur.  
Notebook'tan bağımsız olarak terminalde çalıştırılır:

```bash
pip install streamlit
streamlit run app.py
```

**Deploy (Streamlit Cloud):**
1. Projeyi GitHub'a push'la (`app.py`, `model_classification.pkl`, `scaler.pkl`, `requirements.txt`)
2. [share.streamlit.io](https://share.streamlit.io) → "New app" → repo'yu seç
3. Main file: `app.py` → Deploy!

In [ ]:
app_code = '''
import streamlit as st
import numpy as np
import joblib

model  = joblib.load('model_classification.pkl')
scaler = joblib.load('scaler.pkl')

label_maps = {
    'Gender'             : {'Male': 1, 'Female': 0},
    'Blood Type'         : {'A+': 0, 'A-': 1, 'B+': 2, 'B-': 3, 'AB+': 4, 'AB-': 5, 'O+': 6, 'O-': 7},
    'Medical Condition'  : {'Arthritis': 0, 'Asthma': 1, 'Cancer': 2, 'Diabetes': 3, 'Hypertension': 4, 'Obesity': 5},
    'Insurance Provider' : {'Aetna': 0, 'Blue Cross': 1, 'Cigna': 2, 'Medicare': 3, 'UnitedHealthcare': 4},
    'Admission Type'     : {'Elective': 0, 'Emergency': 1, 'Urgent': 2},
    'Medication'         : {'Aspirin': 0, 'Ibuprofen': 1, 'Lipitor': 2, 'Paracetamol': 3, 'Penicillin': 4},
}
result_map = {0: '🟠 Inconclusive', 1: '🔴 Abnormal', 2: '🟢 Normal'}

st.title('🏥 Hasta Test Sonucu Tahmini')
st.markdown('Hasta bilgilerini girerek test sonucunu tahmin edin.')
st.divider()

col1, col2 = st.columns(2)

with col1:
    age            = st.number_input('Yaş', min_value=1, max_value=120, value=45)
    billing        = st.number_input('Fatura Tutarı (USD)', min_value=0.0, value=15000.0, step=100.0)
    yatis_gun      = st.number_input('Yatış Süresi (gün)', min_value=1, max_value=60, value=5)
    admission_ay   = st.selectbox('Başvuru Ayı', list(range(1, 13)))
    gender         = st.selectbox('Cinsiyet', list(label_maps['Gender'].keys()))
    blood_type     = st.selectbox('Kan Grubu', list(label_maps['Blood Type'].keys()))

with col2:
    condition      = st.selectbox('Hastalık', list(label_maps['Medical Condition'].keys()))
    insurance      = st.selectbox('Sigorta', list(label_maps['Insurance Provider'].keys()))
    admission_type = st.selectbox('Başvuru Tipi', list(label_maps['Admission Type'].keys()))
    medication     = st.selectbox('İlaç', list(label_maps['Medication'].keys()))

st.divider()

if st.button('Tahmin Et', use_container_width=True, type='primary'):
    features = np.array([[
        age,
        yatis_gun,
        billing,
        admission_ay,
        label_maps['Gender'][gender],
        label_maps['Blood Type'][blood_type],
        label_maps['Medical Condition'][condition],
        label_maps['Insurance Provider'][insurance],
        label_maps['Admission Type'][admission_type],
        label_maps['Medication'][medication],
    ]])

    features_sc  = scaler.transform(features)
    prediction   = model.predict(features_sc)[0]
    proba        = model.predict_proba(features_sc)[0]

    st.subheader('Tahmin Sonucu')
    st.markdown(f'### {result_map[prediction]}')

    st.subheader('Olasılık Dağılımı')
    for i, label in result_map.items():
        st.progress(float(proba[i]), text=f'{label}: %{proba[i]*100:.1f}')
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())

print('app.py oluşturuldu!')

In [ ]:
# requirements.txt oluştur (Streamlit Cloud için gerekli)
requirements = """streamlit
scikit-learn
pandas
numpy
joblib
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('requirements.txt oluşturuldu!')
print('\nDeploy için GitHub\'a push edilmesi gereken dosyalar:')
print('  - app.py')
print('  - model_classification.pkl')
print('  - scaler.pkl')
print('  - requirements.txt')